[![Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giordamaug/Relational_EHR_data_to_Longitudinal_Patient_Trajectories/blob/main/run_experiments.ipynb)

## Settings

In [ ]:
dataset = "INA"
results_dir = "experiment_main"
iter = 10
methods = "STATIC STATIC+FREQUENCY"
metrics = ['AUC', 'F1', 'Sensitivity', 'Specificity', 'MCC', 'Brier']
#methods = "STATIC STATIC+BINARY STATIC+FREQUENCY STATIC+EVENT-CNT STATIC+TF-IDF STATIC+BOW STATIC+BiPadLSTM STATIC+Med2Vec STATIC+Dipole STATIC+CEHR-BERT STATIC+DOME STATIC+GRU"

## Download files from Github

In [ ]:
try:
    import google.colab
    %mkdir data
    %mkdir scripts
    %mkdir {results_dir}
    !wget -np https://raw.githubusercontent.com/giordamaug/Relational_EHR_data_to_Longitudinal_Patient_Trajectories/main/data/{dataset}_dataset.json -O data/{dataset}_dataset.json
    !wget -np https://raw.githubusercontent.com/giordamaug/Relational_EHR_data_to_Longitudinal_Patient_Trajectories/main/scripts/models.py -O scripts/models.py
    !wget -np https://raw.githubusercontent.com/giordamaug/Relational_EHR_data_to_Longitudinal_Patient_Trajectories/main/scripts/utils.py -O scripts/utils.py
    !wget -np https://raw.githubusercontent.com/giordamaug/Relational_EHR_data_to_Longitudinal_Patient_Trajectories/main/scripts/classifiers.py -O scripts/classifiers.py
    !wget -np https://raw.githubusercontent.com/giordamaug/Relational_EHR_data_to_Longitudinal_Patient_Trajectories/main/scripts/embedding.py -O scripts/embedding.py
    !wget -np https://raw.githubusercontent.com/giordamaug/Relational_EHR_data_to_Longitudinal_Patient_Trajectories/main/scripts/cv_folding.py -O scripts/cv_folding.py
    !wget -np https://raw.githubusercontent.com/giordamaug/Relational_EHR_data_to_Longitudinal_Patient_Trajectories/main/scripts/configure.py -O scripts/configure.py
    !wget -np https://raw.githubusercontent.com/giordamaug/Relational_EHR_data_to_Longitudinal_Patient_Trajectories/main/scripts/stats.py -O scripts/stats.py
    !wget -np https://raw.githubusercontent.com/giordamaug/Relational_EHR_data_to_Longitudinal_Patient_Trajectories/main/scripts/plotting.py -O scripts/plotting.py
    !wget -np https://raw.githubusercontent.com/giordamaug/Relational_EHR_data_to_Longitudinal_Patient_Trajectories/main/metric_calculations.py -O metric_calculations.py
    !wget -np https://raw.githubusercontent.com/giordamaug/Relational_EHR_data_to_Longitudinal_Patient_Trajectories/main/notebook.py -O notebook.py
    print("Working on Colab!")
except:
    print("Not Working on Colab ... or cannot dwnload Githb files!")
    pass

In [ ]:
%run notebook.py --dataname {dataset} --datafile data/{dataset}_dataset.json --results-dir {results_dir} --methods {methods} --mode embed --iter {iter}

## Calculate metric statistics
with boot resampling

In [ ]:
%run metric_calculations.py --dataset {dataset} --results-dir {results_dir} --methods {methods} --iter {iter}
import pandas as pd
pd.read_csv(f"{results_dir}/result_table_{dataset}_ITER{iter}.csv")

## Calibration plots

In [8]:
import pandas as pd
from scripts.plotting import plot_calibration_curves

summaries = {}
curves = {}
bands = {}
for m in methods.split(" "):
    calibration = pd.read_csv(f"{results_dir}/calibration_{dataset}_{m}_ITER{iter}.csv")
    summaries[m] = pd.read_csv(f"{results_dir}/summary_{dataset}_{m}_ITER{iter}.csv",index_col=0,)
    x = calibration["prob_pred"].to_numpy()
    curves[m] = (x,calibration["prob_true"].to_numpy(),)
    bands[m] = (x,calibration["ci95_low"].to_numpy(),calibration["ci95_high"].to_numpy(),)
    
fig, ax = plot_calibration_curves(
    methods=methods.split(' '),
    summary=summaries,
    curves=curves,
    dataset=dataset,
    bands=bands,
    figsize=(10, 7),
    legend_fontsize=8,
    show_ci_band=False,
)

FileNotFoundError: [Errno 2] No such file or directory: 'experiment_main/calibration_INA_STATIC_ITER10.csv'

## Radar plot

In [ ]:
from scripts.plotting import plot_radar
import pandas as pd
table = pd.read_csv(f"{results_dir}/result_table_{dataset}_ITER{iter}.csv",index_col=0,).drop(columns=['Confusion matrix']).apply(lambda col: col.str[0:5])
table['Brier inv'] = table['Brier'].apply(lambda x: f"{1-float(x)}")
table.drop(columns=['Brier', 'Intercept', 'Slope'], inplace=True)
plot_radar(table,title=f"{dataset}")

## Statistical Analysis

In [1]:
import pickle
import pandas as pd
from scripts.stats import stats_analysis
# Load results
results = {m: {} for m in methods.split(' ')}
for i in range(iter):
    for m in methods.split(' '):
        meth = m.replace('+','_')
        path = (f"{results_dir}/results_{dataset}_{meth}_ITER{i}.pkl")
        with open(path, "rb") as f:
            results[m][i] = pickle.load(f)
friedman_df, ranking_table = stats_analysis(results, ['AUC', 'F1'], methods.split(' '), iter)
friedman_df["p_value"] = pd.to_numeric(friedman_df["p_value"], errors="raise").astype(float)
display(friedman_df)
display(ranking_table)


=== FRIEDMAN TEST ===
Statistic: 28.6383
p-value: 6.04e-07
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 69.1739
p-value: 9.53e-16
➡️ SIGNIFICANT global differences


,statistic,p_value,global_diff
AUC,28.638298,6.043279e-07,YES
F1,69.173913,9.529621e-16,YES


,AUC,F1
STATIC+DOME,1,1
STATIC+FREQUENCY,1,1
STATIC+BINARY,3,3
